# RAG-Enhanced Coaching

**Purpose:** Extends the AI financial coaching system so it can ground its recommendations in a curated collection of financial-education content, not just the user's own numbers.

**Context, for anyone starting here.** This notebook is one piece of a larger AI financial-coaching system; the profile analytics behind it (persona clustering, behavioral scoring, spending forecasts, benchmarking) are produced upstream and simply consumed here as a profile dict. What this notebook adds is the retrieval layer: a small knowledge base of financial-education articles, embedded into a vector store, that the coach searches before answering a question. Combining that retrieved context with the user's own financial profile lets the coach give advice that is both grounded in source material and specific to the person asking.

**Design Notes:**

1. Two knowledge domains are seeded to start: emergency funds and debt reduction
  - Both are common early-stage coaching topics, and both have a natural progression (starter fund up through one, three, and six months of expenses; smallest-balance-first versus highest-interest-first repayment) that benefits from being grounded in a fixed reference rather than left to the model's general knowledge
2. Coverage is intentionally partial
  - Questions outside the knowledge base fall back to the model's general financial knowledge rather than being blocked, so the coach still answers, just without a source-grounded citation
3. The knowledge base and the coaching logic are built as two separable stages (§1, §2), so new domains can be added to the vector store without touching the coaching pipeline


In [1]:
# Standard Libraries
import os
import re

# Data Analysis
import pandas as pd
from IPython.display import display

# Environment Configuration
from dotenv import load_dotenv

# LangChain
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# OPENAI_API_KEY
load_dotenv(override = True)


True

---
## 1 · Knowledge Base Processing & Vector Store

**Purpose:** Turns the knowledge base's source files into searchable embeddings, so the coaching pipeline in §2 can retrieve grounding context instead of relying on the model's unverified general knowledge.

**Design Notes:**

1. One unified Chroma collection (financial_coach) rather than one per topic
  - Retrieval stays cross-domain by construction: as more topics are added, a question can pull relevant context from any of them without the pipeline needing to know which collection to search
2. Each source file carries its own identity in its header: a domain, a set of tags, and ##-delimited subsections
  - Splitting on those subsections turns each file into topic-sized chunks rather than embedding the whole document at once, so retrieval returns the specific concept a question is about, not the entire article
  - Every chunk is tagged with its source filename, so a document can be replaced (replace_knowledge_document) by deleting only the chunks with that source and re-adding the updated file, without rebuilding the collection
3. Coverage today is two files; the pipeline itself is domain-agnostic and expects new files to follow the same header/section convention


In [2]:
# Create Vector Store
embeddings = OpenAIEmbeddings()

vectorstore = Chroma(
    collection_name = 'financial_coach',
    embedding_function = embeddings,
    persist_directory = 'chroma_db',
)


In [3]:
# Knowledge Base Processing Pipeline
def load_knowledge_document(file_path):
    # Load Document
    loader = TextLoader(file_path)
    text = loader.load()[0].page_content

    # Domain
    domain_match = re.search(r'Domain:\s*(.+)', text)
    domain = domain_match.group(1).strip() if domain_match else 'Unknown'

    # Tags
    tags_match = re.search(r'Tags:\s*(.*?)\n\nContent Type:', text, re.DOTALL)
    tags = []
    if tags_match:
        tags = [
            tag.strip()
            for tag in tags_match.group(1).replace('\n', ' ').split(',')
            if tag.strip()
        ]

    # Source File
    source = os.path.basename(file_path)

    # Split Into Sections
    sections = re.split(r'\n##\s+', text)[1:]

    # Build Chunks
    chunks = []
    for section in sections:
        lines = section.strip().split('\n')
        topic = lines[0].strip()

        content = '\n'.join(lines[1:]).strip()
        content = re.sub(r'\n?-{3,}\s*$', '', content).strip()

        chunks.append(Document(
            page_content=content,
            metadata={
                'domain': domain,
                'topic': topic,
                'source': source,
                'tags': tags,
            },
        ))

    print(f'{source}: {len(chunks)} chunks created.')
    return chunks


# Add New Document
def add_knowledge_document(file_path, vectorstore):
    chunks = load_knowledge_document(file_path)
    vectorstore.add_documents(chunks)

    filename = os.path.basename(file_path)
    print(f'{filename}: {len(chunks)} chunks added.')
    print(f'Total Chunks: {vectorstore._collection.count()}\n')


# Replace Existing Document
def replace_knowledge_document(file_path, vectorstore):
    source = os.path.basename(file_path)

    # Remove Existing Chunks, Reload Updated Document
    vectorstore._collection.delete(where = {'source': source})
    add_knowledge_document(file_path, vectorstore)
    print(f'{source} replaced.\n')


In [4]:
# Add Documents
add_knowledge_document('knowledge_base/01_emergency_fund.txt', vectorstore)
add_knowledge_document('knowledge_base/02_debt_reduction.txt', vectorstore)


01_emergency_fund.txt: 10 chunks created.
01_emergency_fund.txt: 10 chunks added.
Total Chunks: 10

02_debt_reduction.txt: 14 chunks created.
02_debt_reduction.txt: 14 chunks added.
Total Chunks: 24



In [5]:
# Validate Vector Store
all_docs = vectorstore.get()
metadata_df = pd.DataFrame(all_docs['metadatas'])

display(metadata_df['source'].value_counts())


source
02_debt_reduction.txt    14
01_emergency_fund.txt    10
Name: count, dtype: int64

**Observations:**

1. 01_emergency_fund.txt splits into 10 chunks and 02_debt_reduction.txt into 14, one per ## subsection in each source file
2. Chunk counts track each file's outline directly: adding or removing a ## subsection changes the chunk count on the next load, nothing else needs to change


In [6]:
# Inspect Sample Chunk
all_docs = vectorstore.get()

for doc, meta in zip(all_docs['documents'], all_docs['metadatas']):
    if meta['source'] == '01_emergency_fund.txt':
        print('Metadata:')
        print(meta)

        print('\nContent:')
        print(doc)
        
        break


Metadata:
{'domain': 'Emergency Fund', 'source': '01_emergency_fund.txt', 'topic': '1.1 Definition', 'tags': ['Emergency Fund', 'Emergency Savings', 'Financial Stability', 'Financial Resilience', 'Unexpected Expenses', 'Personal Finance']}

Content:
Summary:

An emergency fund is dedicated savings used to cover unexpected expenses and temporary income disruptions.

Keywords:
emergency fund, emergency savings, financial safety net,
unexpected expenses, income disruption, financial resilience

Definition:

An emergency fund is a dedicated pool of savings set aside specifically to cover unexpected financial emergencies or temporary income disruptions without relying on credit cards, loans, investments, or retirement accounts. It serves as a financial safety net that helps individuals and households maintain financial stability during unforeseen events such as job loss, medical emergencies, or essential home and vehicle repairs. Unlike savings intended for planned expenses, an emergency fu

---
## 2 · RAG-Enhanced Financial Coaching

**Purpose:** Answers a user's personal-finance question by combining retrieved knowledge-base context (when the topic is covered) with their own financial profile, so the response is both source-grounded and personalized.

**Design Notes:**

1. Every question is classified first, into emergency_fund, debt_reduction, other_finance, or not_finance
  - The two knowledge-base topics retrieve grounding context via vector search before generation; other_finance falls back to the model's general knowledge instead of being blocked; not_finance is declined outright, keeping the coach inside personal-finance topic boundaries
2. The user's financial profile (income, spending, debt, emergency fund status, and forecast) is folded into every response regardless of which path was used, so advice is never generic even when it isn't knowledge-base-grounded
3. Grounding responses in curated content for the two covered topics reduces hallucination risk on exactly the questions where a wrong answer, an invented emergency fund target, for example, would be most consequential


### 2.1 Test Profile

**Purpose:** Stands in for the upstream profile analytics (see context above) with one representative user: a "Near Successful" persona carrying a thin surplus, no emergency fund, and modest debt.


In [7]:
# Test Profile: Near Successful Persona
profile = {
    'persona': {
        'name': 'Near Successful',
        'cluster_id': 1,
        'description': (
            'Users whose spending patterns closely resemble financially successful.'
            'users but who generate insufficient monthly surplus.'
        ),
    },

    'behavioral_alignment': {
        'success_similarity': 0.8259,
        'score': 82.6,
    },

    'financial_capacity': {
        'monthly_income': 4811.00,
        'avg_monthly_spend': 4731.66,
        'monthly_surplus': 79.34,
        'surplus_ratio': 0.0165,
        'score': 1.6,
    },

    'financial_stability': {
        'total_debt': 2262.00,
        'debt_to_income': 0.0392,
    },

    'emergency_fund': {
        'balance': 0.00,
        'target': 14433.00,
        'progress': 0.0669,
    },

    'forecasting': {
        'predicted_next_month_spending': 4942.56,
        'forecast_surplus': -131.56,
    },

    'benchmark': {
        'monthly_savings_opportunity': 381.00,
    },

    'spending_categories': [
        {'category': 'Shopping', 'amount': 650.00},
        {'category': 'Transportation', 'amount': 525.00},
        {'category': 'Food Discretionary', 'amount': 480.00},
    ],
}


### 2.2 Coaching Pipeline

**Purpose:** Classifies the question, retrieves knowledge-base context when the topic is covered, and generates a personalized response grounded in whichever source, knowledge base or general knowledge, applies.

**Design Notes:**

1. Classification and generation both run on gpt-4o-mini at a moderate temperature (0.5), favoring consistent, on-topic responses over creative variation
2. The generation prompt states explicitly that knowledge-base context, when present, takes priority over the model's own knowledge and must not be contradicted
3. Responses are capped at 250 words to keep coaching output actionable rather than exhaustive


In [8]:
# Coaching Pipeline: Classify, Retrieve, Generate
llm = ChatOpenAI(model = 'gpt-4o-mini', temperature = 0.5)


# Topic Classification
def classify_question(question):
    
    prompt = ChatPromptTemplate.from_template('''
    
        You are a financial topic classifier.

        Classify the user's question into ONE category:
        - emergency_fund
        - debt_reduction
        - other_finance
        - not_finance

        Return ONLY the category name.

        User Question: {question}
        
    ''')

    chain = prompt | llm | StrOutputParser()
    
    return chain.invoke({'question': question}).strip().lower()


# Retrieve Knowledge Base Context
def retrieve_context(question, vectorstore, k=4):
    results = vectorstore.similarity_search(question, k=k)
    context = '\n\n'.join(doc.page_content for doc in results)
    sources = list({doc.metadata.get('source', 'Unknown') for doc in results})
    
    return context, sources


# Profile Summary
def profile_summary(profile):
    
    return f'''
    
        Persona: {profile['persona']['name']}

        Monthly Income: ${profile['financial_capacity']['monthly_income']:,.0f}
        Monthly Spending: ${profile['financial_capacity']['avg_monthly_spend']:,.0f}
        Monthly Surplus: ${profile['financial_capacity']['monthly_surplus']:,.0f}

        Emergency Fund Balance: ${profile['emergency_fund']['balance']:,.0f}
        Emergency Fund Target: ${profile['emergency_fund']['target']:,.0f}

        Total Debt: ${profile['financial_stability']['total_debt']:,.0f}
        Debt-to-Income Ratio: {profile['financial_stability']['debt_to_income']:.1%}

        Forecast Surplus: ${profile['forecasting']['forecast_surplus']:,.0f}

        Potential Monthly Savings Opportunity:
        ${profile['benchmark']['monthly_savings_opportunity']:,.0f}
        
    '''


# Financial Coach
def ask_financial_coach(profile, question, vectorstore):
    
    category = classify_question(question)

    # Non-Finance Questions
    if category == 'not_finance':
        return 'I can only assist with personal finance related questions.'

    # Knowledge Base Topics
    if category in ('emergency_fund', 'debt_reduction'):
        context, sources = retrieve_context(question, vectorstore)
        source_type = 'Knowledge Base'

    # Other Finance Topics
    else:
        context, sources, source_type = '', [], 'General Financial Knowledge'

    prompt = ChatPromptTemplate.from_template('''
    
        You are a professional financial coach.

        User Financial Profile: {profile}

        Information Source: {source_type}

        Knowledge Base Context: {context}

        User Question: {question}

        Requirements:
        - Answer only personal finance questions.
        - If Knowledge Base Context is provided,
          rely primarily on that information.
        - Do not invent facts that contradict
          the knowledge base.
        - If no knowledge base context is provided,
          answer using general financial knowledge.
        - Personalize advice using the user's profile.
        - Focus on practical, actionable guidance.
        - Maintain a professional and supportive tone.
        - Limit responses to 250 words or less.
        
    ''')

    chain = prompt | llm | StrOutputParser()
    
    return chain.invoke({
        'profile': profile_summary(profile),
        'source_type': source_type,
        'context': context,
        'question': question,
    })


In [9]:
# Quick Test
response = ask_financial_coach(
    profile = profile,
    question = 'What is the best way to start an emergency fund?',
    vectorstore = vectorstore,
)

print(response)


To start building your emergency fund, follow these actionable steps tailored to your financial profile:

1. **Set a Target**: Aim for a starter emergency fund of $500 to $1,000. This amount can help cover many common unexpected expenses.

2. **Identify Savings Opportunities**: You have a potential monthly savings opportunity of $381. Consider setting aside a portion of this each month specifically for your emergency fund.

3. **Automate Savings**: Set up an automatic transfer from your checking account to a high-yield savings account dedicated to your emergency fund. This makes saving effortless and ensures you prioritize this goal.

4. **Cut Non-Essential Spending**: Review your monthly spending of $4,732. Identify areas where you can reduce expenses, even slightly, to increase your monthly surplus and contribute more to your emergency fund.

5. **Track Progress**: Monitor your emergency fund balance regularly to stay motivated. Celebrate milestones, such as reaching $500 or $1,000.


### 2.3 Example Behaviors

**Purpose:** Walks through the four classification paths, knowledge-base retrieval (two topics), general financial knowledge, and topic-boundary enforcement, against the same test profile.

**Observations:**

1. The two knowledge-base topics return specific, source-grounded guidance, a 500 - 1,000 starter emergency fund target and a snowball-versus-avalanche comparison, drawn from the retrieved chunks rather than the model's own knowledge
2. The other_finance question (Roth IRA) is still answered and still personalized to the profile, just without knowledge-base grounding or sources
3. The not_finance question is declined outright, holding the topic boundary regardless of profile context


In [10]:
# Example: Emergency Fund (Knowledge Base)
response = ask_financial_coach(
    profile, 'How much should I keep in my emergency fund?', vectorstore
)

print(response)


Given your financial profile, it's essential to start building an emergency fund, especially since you currently have none. A good initial target would be to establish a starter emergency fund of $500 to $1,000. This amount can help cover many common unexpected expenses, such as car repairs or medical bills.

Since your monthly spending is $4,732, aiming for one month of essential living expenses would be a prudent next step. This would be approximately $4,732, providing a more substantial safety net against temporary income disruptions.

To achieve this, you can utilize the potential monthly savings opportunity of $381. By allocating this amount toward your emergency fund, you could reach your starter fund goal in about 1.5 to 2.5 months, depending on whether you aim for $500 or $1,000.

Once you've established this initial fund, you can gradually work toward a more robust emergency fund of three months' worth of expenses, which would be roughly $14,196. This level of savings is often

In [11]:
# Example: Debt Reduction (Knowledge Base)
response = ask_financial_coach(
    profile, 'Should I use debt snowball or debt avalanche?', vectorstore
)

print(response)


Given your financial profile, both the debt snowball and debt avalanche methods have their merits, but your choice should align with your personal preferences and financial goals.

**Debt Snowball Method:** This approach focuses on paying off your smallest debt first. Since your total debt is relatively low at $2,262, this method can provide quick wins and motivation as you eliminate each debt. It may help you feel more accomplished and encourage you to stay committed to your repayment plan.

**Debt Avalanche Method:** This strategy prioritizes paying off the debt with the highest interest rate first. If any of your debts carry high interest, this method can save you money on interest payments over time. However, if your debts are similar in interest rates, the snowball method might be more motivating for you.

Given your forecast surplus of -$132, it's crucial to manage your spending closely. You have a potential savings opportunity of $381 monthly, which can be allocated toward debt 

In [12]:
# Example: Other Finance (General Knowledge)
response = ask_financial_coach(
    profile, 'How does a Roth IRA work?', vectorstore
)

print(response)


A Roth IRA (Individual Retirement Account) is a retirement savings account that allows you to contribute after-tax income, meaning you've already paid taxes on the money you put in. Here’s how it works:

1. **Contributions**: You can contribute up to a certain limit each year ($6,500 for 2023, or $7,500 if you're 50 or older). Since your monthly surplus is currently $79, you may want to consider how you can allocate some of your potential monthly savings opportunity of $381 towards this account.

2. **Tax Benefits**: The key advantage of a Roth IRA is that your investments grow tax-free, and you can withdraw your contributions (not earnings) at any time without penalty. After age 59½, you can withdraw earnings tax-free as long as the account has been open for at least five years.

3. **Eligibility**: Your ability to contribute may be limited based on your income. For 2023, single filers with a modified adjusted gross income (MAGI) of up to $138,000 can contribute the full amount, with 

In [13]:
# Example: Not Finance (Topic Boundary)
response = ask_financial_coach(
    profile, 'How do I make lasagna?', vectorstore
)

print(response)


I can only assist with personal finance related questions.
